## Configuration and Imports
Set up the Ollama API connection and define the models to use for chat and embeddings.

**Key Definitions:**
- `HOST`: Base URL for Ollama API (localhost:11434)
- `MODEL_NAME`: LLM model for chat and reasoning (netops-llama3.2-vision)
- `EMBED_MODEL_NAME`: Embedding model for vectorizing text (embeddinggemma)
- `ollama_get()`: Helper function to send GET requests to Ollama API

**Purpose:**
Initialize the connection to locally-running Ollama models and verify availability. This cell establishes the foundation for the RAG system by connecting to both the chat model (for generating responses) and the embedding model (for semantic search). The verification step ensures the models are accessible before proceeding with document processing.

In [ ]:
import json, urllib.request, urllib.error, os, math, re
from dataclasses import dataclass
from typing import List, Tuple, Dict

HOST = "http://localhost:11434"

MODEL_NAME = "netops-llama3.2-vision:latest"
EMBED_MODEL_NAME = "embeddinggemma:latest"

def ollama_get(path: str, timeout=10) -> dict:
    request = urllib.request.Request(HOST + path, method="GET")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))
    
print("Ollama version:", ollama_get("/api/version"))
tags = ollama_get("/api/tags")
print("Available models:", [m["name"] for m in tags.get("models", [])][:10], "...")


## API Functions for Ollama
Define utility functions to communicate with the Ollama API for embeddings and chat.

**Key Functions:**
- `ollama_post()`: Send POST requests to Ollama API with JSON payloads
- `embed_text()`: Convert text into dense vector embeddings for semantic search
- `chat_user_only()`: Send a user message to the LLM and receive a response with temperature control

**Purpose:**
Create reusable API wrapper functions for interacting with Ollama models. The `embed_text()` function transforms text into high-dimensional vectors that capture semantic meaning, enabling similarity-based search. The `chat_user_only()` function provides a simple interface to get LLM responses with configurable temperature (controls randomness/creativity of responses).

In [ ]:
def ollama_post(path: str, payload: dict, timeout=120) -> dict:
    request = urllib.request.Request(
        HOST + path,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))
    
def embed_text(text: str) -> List[float]:
    response = ollama_post("/api/embed", {"model": EMBED_MODEL_NAME, "input": text}, timeout=120)
    return response.get("embedding", [])

def chat_user_only(user_content: str, temperature: float = 0.2) -> str:
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": user_content}],
        "stream": False,
        "options": {"temperature": temperature}
        }
    response = ollama_post("/api/chat", payload, timeout=180)
    return response.get("message", {}).get("content", "")

## Text Processing and Chunking
Prepare documents for embedding by normalizing text and splitting into manageable chunks.

**Key Functions:**
- `normalize_text()`: Clean up whitespace and normalize text formatting
- `chunk_text()`: Split documents into overlapping chunks to preserve context (size=1000 chars with 200 char overlap)

**Purpose:**
Transform raw documents into consistent, embedding-friendly chunks that can be efficiently vectorized and retrieved. The overlap between chunks (200 chars) ensures that important context isn't lost at chunk boundaries, improving retrieval quality for questions that might span multiple segments. This chunking strategy balances capturing sufficient context while keeping embeddings computationally efficient.

In [ ]:
def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text.strip())

def chunk_text(text: str, chunk_size: int = 1000, overlap: int=200) -> List[str]:
    """
    Chunk by characters with overlap (simple + reliable).
    For docs/runbooks/logs, we can chunk by lines or sections instead, but for general text this is more robust.
    """
    text = normalize_text(text)
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

## Load Source Documents
Read documents from the assets/documents directory (supports .txt, .md, .log, .csv, .jsonl, .yml).

**Key Outputs:**
- `DOCUMENTS`: List of document contents loaded from files
- `DOCUMENT_SOURCEs`: List of file paths corresponding to each document (for source attribution)

**Purpose:**
Automatically discover and load all relevant documents from the designated directory to build the knowledge base for the RAG system. Supports multiple file formats commonly used in network operations (logs, markdown runbooks, YAML configs, etc.). The source tracking enables proper attribution in generated responses. Non-empty documents are filtered to ensure quality, and error handling allows the loading process to continue even if individual files fail.

In [ ]:
import pathlib

DOCUMENTS_DIR = pathlib.Path("../assets/documents")
DOCUMENT_EXTENSIONS = [".txt", ".md", ".log", ".csv", ".jsonl", ".yml"]  # You can add more extensions if needed
DOCUMENT_PATH = [p for p in DOCUMENTS_DIR.glob("*") if p.is_file() and p.suffix.lower() in DOCUMENT_EXTENSIONS]
DOCUMENTS = []
DOCUMENT_SOURCEs = []

for doc_path in sorted(DOCUMENT_PATH):
    try:
        text = doc_path.read_text(encoding="utf-8", errors="replace")
        if text.strip():  # Only include non-empty documents
            DOCUMENTS.append(text)
            DOCUMENT_SOURCEs.append(str(doc_path))
    except Exception as e:
        print(f"Error reading {doc_path}: {e}")

print(f"Loaded {len(DOCUMENTS)} documents from {DOCUMENTS_DIR}/")
print("Sample document sources:", DOCUMENT_SOURCEs[:5], "...")


# For demonstration, we can embed some sample documents (e.g. runbook sections, troubleshooting checklists) and store them in-memory.
# If you don't currently have any available documents you want to add, you can use the following sample text snippets (these are just examples and can be replaced with your actual documents),
# Just uncomment the below block and comment out the above code that loads documents from the directory, and it will use these sample documents instead of loading from the directory.
#
#  DOCUMENTS = [
# """
# Title: BGP Neighbor Down - Core Checklist
# - Confirm physical link status and errors (interfaces counters).
# - Check BGP hold timers and keepalives; 'hold time expired' often indicates loss/congestion.
# - Verify MTU on the path if sessions flap during peak.
# - Correlate with interface link events (down/up).
# - Validate reachability (ping, traceroute) and control-plane policing.
# """,
# """
# Title: One-Way Audio Troubleshooting (SIP/RTP)
# - Confirm SDP offers correct IP/port; check NAT and ALG behavior.
# - Verify RTP is flowing both directions (packet captures, SBC stats).
# - Look for codec mismatch, asymmetric routing, firewall pinholes timing out.
# - For intermittent issues, correlate with jitter/loss; enable RTCP stats if available.
# """,
# """
# Title: QoS Quick Notes
# - For voice, prioritize RTP (DSCP EF) and SIP signaling (CS3/AF31 per policy).
# - Validate marking at ingress and no remarking in transit.
# - Check queue drops and shaping policies on egress.
# """
# ]


## Build Vector Index
Create embeddings for all document chunks and store them in an in-memory vector index for semantic search.

**Key Components:**
- `DocumentChunk`: Data class storing document metadata (doc_id, chunk_id), text, and embeddings
- `cosine_similarity()`: Compute similarity between query and stored embeddings using dot product normalized by vector magnitudes
- `index`: In-memory vector store with embeddings for all chunks

**Purpose:**
Process all documents by chunking them (500 chars with 100 char overlap) and generating embeddings for each chunk using the embedding model. The resulting vector index enables fast semantic search - finding documents based on meaning rather than exact keyword matches. Each chunk maintains references to its source document for attribution. This indexing step is the core of the RAG retrieval mechanism.

In [ ]:
@dataclass
class DocumentChunk:
    doc_id: int
    chunk_id: int
    text: str
    embedding: List[float]

def cosine_similarity(a: List[float], b: List[float]) -> float:
    dot_product = sum(x*y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x*x for x in a))
    norm_b = math.sqrt(sum(y*y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

index: List[DocumentChunk] = []

for doc_id, doc in enumerate(DOCUMENTS):
    chunks = chunk_text(doc, chunk_size=500, overlap=100)
    for chunk_id, chunk in enumerate(chunks):
        embedding = embed_text(chunk)
        index.append(DocumentChunk(doc_id=doc_id, chunk_id=chunk_id, text=chunk, embedding=embedding))

print(f"Indexed {len(DOCUMENTS)} docs in {len(index)} chunks with embeddings model '{EMBED_MODEL_NAME}'.")

## Semantic Search and Retrieval
Find the most relevant document chunks for a given query using embedding similarity.

**Key Function:**
- `retrieve_relevant_chunks()`: Embed the query and find top-k most similar chunks from the index using cosine similarity scoring

**Purpose:**
Perform semantic search over the document collection to find the most relevant context for answering questions. The query is embedded using the same model as the documents, then compared against all indexed chunks using cosine similarity. This approach finds semantically similar content even when the query uses different wording than the source documents, enabling more flexible and natural question-answering. Results are ranked by similarity score and the top-k chunks are returned.

In [ ]:
def retrieve_relevant_chunks(query: str, top_k: int = 3) -> List[Tuple[DocumentChunk, float]]:
    query_embedding = embed_text(query)
    scored_chunks = []
    for chunk in index:
        score = cosine_similarity(query_embedding, chunk.embedding)
        scored_chunks.append((chunk, score))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)
    return scored_chunks[:top_k]

test = retrieve_relevant_chunks("What are the common causes for BGP neighbor down and how to troubleshoot?", top_k=2)
for chunk, score in test:
    print(f"Score: {score:.4f}, DocID: {chunk.doc_id}, ChunkID: {chunk.chunk_id}, Text: {chunk.text[:100]}...")

## Generate RAG Response
Combine retrieved context with the user's question and prompt the LLM for a structured answer.

**Key Function:**
- `generate_rag_response()`: Retrieve relevant chunks, format them as context, and send to the LLM with a structured prompt template to generate answers with bullet points, next steps, verification steps, and source attribution

**Purpose:**
Implement the complete RAG (Retrieval-Augmented Generation) pipeline: retrieve the most relevant document chunks for a question, format them as context with similarity scores and source attribution, then prompt the LLM to generate a structured response based solely on the retrieved information. The structured prompt ensures responses include actionable troubleshooting steps, verification procedures, and proper source citations, making it particularly useful for network operations and technical support scenarios.

In [ ]:
RAG_PROMPT_TEMPLATE = """
Use only the following retrieved document chunks to answer the question. If the retrieved information is insufficient to answer the question, say you don't know. Always use all relevant information from the retrieved chunks in your answer and suggest next steps if applicable.
Return:

1) A concise answer in bullet points to the question based on the retrieved information.
2) Next steps or suggestions for further investigation (Ordered list).
3) Verification steps to confirm the issue or validate the solution.
4) Sources of the retrieved information (list the document sources corresponding to the retrieved chunks).

Keep it concise and focused on the question. Do not include any information that is not directly relevant to the question.
"""

def generate_rag_response(question: str, top_k: int = 4) -> str:
    top_chunks = retrieve_relevant_chunks(question, top_k=top_k)
    context_block = []
    for rec, score in top_chunks:
        context_block.append(
            f"[{rec.doc_id}-{rec.chunk_id}] (score: {score:.4f})\n(Source: {DOCUMENT_SOURCEs[rec.doc_id]}\n{rec.text}\n)"
        )
    prompt = f"""Question: {question}
Retrieved Context:
{'\n'.join(context_block)}

{RAG_PROMPT_TEMPLATE}
"""
    response = chat_user_only(prompt, temperature=0.2)
    return response

# print(generate_rag_response("What are the common causes for BGP neighbor down and how to troubleshoot?", top_k=4))
print(generate_rag_response("BGP neighbor down with high CPU usage and memory utilization events. What could be the cause and what should I do first?", top_k=4))